```bash
# Write a match query using the _search API to search the blogs index for Steve in the authors.first_name field. How many hits does the query return?
GET blogs/_search
{
  "query": {
    "match": {
      "authors.first_name": "Steve"
    }
  }
}
```

```
{
  "took": 7,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 33,
      "relation": "eq"
    },
    "max_score": 5.303483,
    "hits": [
      {
        "_index": "blogs",
        "_id": "66f3f482c5f2efa0e7a3474a",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Machine Learning for Nginx Logs - Identifying Operational Issues with Your Website | Elastic Blog",
          "locale": "pt-br,es-mx,en-us",
          "content": """26 September 2017 Machine Learning for Nginx Logs - Identifying Operational Issues with Your Website By Steve Dodson Share Share on Twitter Share on Facebook Share on LinkedInr Editor's Note (August 3, 2021): This post uses deprecated features. Please reference the map custom regions with reverse geocoding documentation for current instructions. Getting insight from nginx log files can be complicated. This blog shows how machine learning can be used to automatically extract operational insights from large volumes of nginx log data. Overview Data science can be a complicated, experimental process where it is easy to get lost in the data , or the counter-intuitiveness of statistics . Therefore, a key design goal for the Machine Learning group at Elastic is to develop tools that empower a wide spectrum of users to get insight out of Elasticsearch data. This lead to us to develop features such as " Single Metric Job " and " Multiple Metric Job " wizards in X-Pack Machine Learning, and we are planning to simplify analysis and configuration steps even more in upcoming releases. In parallel to these wizards, we are also planning to shrink-wrap job configurations on known Beats and Logstash data sources. For example, if you are collecting data with the Filebeat NGINX module , we can provide a set of shrink-wrapped configurations and dashboards to help users apply machine learning to their data. These configurations are also aimed at showing how we develop Machine Learning configurations internally based on our experience. Help us prioritize the next set of modules that should include preconfigured machine learning jobs by filling out this short survey . The details of how to install these configurations will be covered in a subsequent blog. This blog is aimed at describing the use cases and configurations. Use Case Notes The configuration options for X-Pack Machine Learning are extensive, and often new users are tempted to start with complex configurations and select large numbers of attributes and series. These types of configurations can be very powerful and expressive, but require care as the results can be difficult to interpret. We therefore recommend that users start with simple, well-defined use cases, and build out complexity as they become more familiar with the system. (Note, often the best initial use cases come from automating anomaly detection on charts on the Operations teams core dashboards.) Example Data Description The data used in these examples is from a production system consisting of 4 load balanced nginx web servers. We analysed 3 months data (~29,000,000 events, ~1,100,000 unique visitors, ~29GB data). Note, the data shown here has been anonymised. nginx log format : '"$http_x_forwarded_for" $remote_addr - [$time_local] "$request" $status $body_bytes_sent "$http_referer" "$http_user_agent"'; Sample log message: "2021:0eb8:86a3:1000:0000:9b3e:0370:7334 10.225.192.17 10.2.2.121" - - [30/Dec/2016:06:47:09 +0000] "GET /test.html HTTP/1.1" 404 8571 "-" "Mozilla/5.0 (compatible; Facebot 1.0; https://developers.facebook.com/docs/sharing/webmasters/crawler)" Once processed by Filebeat's NGINX module configuration, we get the following JSON document in Elasticsearch: ... { "nginx" : { "access" : { "referrer" : "-", "response_code" : "404", "remote_ip" : "2021:0eb8:86a3:1000:0000:9b3e:0370:7334", "geoip" : { "continent_name" : "Europe", "country_iso_code" : "PT", "location" : { "lon" : -10.23057, "lat" : 34.7245 } }, "method" : "GET", "user_name" : "-", "http_version" : "1.1", "body_sent" : { "bytes" : "8571" }, "remote_ip_list" : [ "2021:0eb8:86a3:1000:0000:9b3e:0370:7334", "10.225.192.17", "10.2.2.121" ], "url" : "/test.html", "user_agent" : { "major" : "1", "minor" : "0", "os" : "Other", "name" : "Facebot", "os_name" : "Other", "device" : "Spider" } } } }... Use Case 1: Changes in Website Visitors Operationally, system issues are often reflected in changes in visitor rate. For example, if the visitor rate declines significantly in a short period of time, it is likely that there is a system issue with the site. Simple ways to understand changes in visitor rate are to analyse overall event rate, or the rate number of distinct visitors. Job 1.1: Low Count of Website Visitors This job can simply be configured using the 'Single Metric Job' wizard: Job configuration summary: This analysis shows a significant anomaly on February 27th where the total event rate drops significantly: (Note this analysis of the 29,000,000 events took a total of 16s on a m4.large AWS instance) Job 1.2: Low Count of Unique Website Visitors Event counts can be strongly influenced by bots or attackers, and so a more consistent feature to analyse the number of unique website visitors. Again this can simply be configured using the 'Single Metric Job' wizard: Again there is a significant anomaly on February 27th where the number of unique visitors per 15m drops from a typical 1487 to 86: Combining Job 1.1 and 1.2: Using the Anomaly Explorer the results from both jobs can be temporary correlated to give an 'Overall' view into the anomalousness of the system based on these features: This clearly shows in a single view, that there was a significant anomaly on February 27th between 10:00-12:00 where the total event rate dropped, and the number of unique visitors dropped. The operations team confirmed the site had significant issues at this time due to a prior configuration change in the CDN. Unfortunately, they didn't detect the user impact until 11:30 (due to internal users on Slack complaining), whereas with ML they would have been alerted at 10:00 when the issue occurred. This analysis can be combined with alerting to give operations teams early insights into changes in system behaviour. Use Case 2: Changes in Website Behaviour Once simple behaviours are analysed, next steps are often to analyse more complex features. For example, changes in event rates of the different HTTP status codes returned by the webserver can often indicate changes in system behaviour or unusual clients: This use case is more complex as it involves analysing multiple series concurrently, but it can again be simply configured using the " Multiple Metric Job " wizard: Results show some significant changes in the different response codes: In particular, again on February 27th there is a significant change in behaviour of response_code 404, 301, 306 and 200. Zooming in on 404s show some significant anomalies: The first highlighted anomaly is attributed to a specific IP address as nginx.access.remote_ip is defined as an influencer (more on this in a later blog). The second highlighted anomaly represents a significant overall change in 404 behaviour. The increase in 404s on February 27th was again a new insight for the operations team, and represented a large number of dead links that had been introduced by the configuration change. Use Case 3: Unusual Clients Website traffic generally consists of a combination of normal usage, scanning by bots and attempted malicious activity. Assuming the majority of clients are normal, we can use population analysis to detect significant attacks or bot activity. The number of pages a normal user requests in a 5-minute window can be limited by how fast they can manually click website pages. Automated processes can scan 1000s of pages a minute, and attackers can simply flood a site with requests. There are a number of features we could use to differentiate traffic types, but in the first instance, event rate and number of distinct URL rate by a client can highlight unusual client activity. In this case, advanced job configuration is used to configure 2 population jobs: Job 3.1: Detect unusual remote_ips - high request rates Looking at unusually high event rate for a client (nginx access remote ip high_count) we get: This shows a number of anomalous clients. For example, 185.78.31.85 seems to be anomalous over a long time period: Drilling into a dashboard that summarises this interaction: This shows that this IP address has repeatedly hit the root URL (/) an unusually large number of times in a short time period, and that this behaviour continues for several days. Job 3.2: Detect unusual remote_ips - high request rates Looking at unusually high distinct count of URL rate for a client (nginx access remote ip high dc url) we get: Again, this shows a number of unusual clients. Drilling into 72.57.0.53 shows a client accessing > 12000 distinct URLs in a short period. Drilling into a dashboard that summarises this interaction: This shows this client is attempting a large number of unusual URLs consistent with path traversal types of attack. Both these jobs provide real-time visibility into unusual clients accessing a website. Web traffic is often skewed by bots and attackers, and differentiating these clients can help administrators understand behaviours such as: What types of attack the site is subjected to Whether bots are successful accessing the entire site What 'normal' usage is Summary This blog attempts to show how X-Pack ML can provide insights into website behaviour. In upcoming Elastic Stack releases these types of configurations and dashboards will be available to end users as easily installed packages. This should empower users with proven tested configurations and also show users recommended types of configurations to copy and extend.""",
          "seo": "",
          "url": "https://www.elastic.co/blog/machine-learning-for-nginx-logs",
          "versions": [
            "5.6"
          ],
          "publish_date": "2017-09-26T21:20:13.000Z",
          "authors": [
            {
              "last_name": "Dodson",
              "title": "Steve Dodson",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Distinguished Engineer II"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f5523bc5f2eff540a80779",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword",
          "seo.keyword"
        ],
        "_source": {
          "title": "The Elastic (ELK) Stack: Free. Open. Limitless. | Elastic Blog",
          "locale": "de-de,fr-fr,ja-jp,ko-kr,zh-cn,pt-br,es-mx,en-us",
          "content": "06 April 2020 News The Elastic Stack: Free. Open. Limitless. By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr From the very beginning, the Elastic Stack — Elasticsearch, Kibana, Beats, and Logstash — has been free and open. Our approach is not only to make our technology stack available for free, but to make it open — housed in public repositories and developed through a transparent approach with direct involvement from the community. Two simple principles — free and open — broke down barriers and enabled many amazing things. Vibrant community Combine great technology with free distribution and open development and you get a vibrant community of doers. Free lowers barriers to adoption, and open development fosters collaboration, contribution, and creativity. Anyone in the world can download the Elastic Stack and get started immediately — whether they choose to run it on a laptop to develop a new search-powered application or in a data center to monitor infrastructure and protect against security threats. They can see (and contribute to) the code, share feedback, questions, or requests directly with our engineering team, and engage with their peers in the community. This is a powerful force multiplier. Better products. Newer directions. Combine free and open with a creative, passionate, and engaged community, and not only does it make the products better, but the community often blazes the trails that take the products in new and interesting directions. Our community is a source of constant inspiration for us, and is the source of so many of the great ideas that move us forward. In the early days of Elasticsearch, the ingenuity of the community gave rise to Logstash, Kibana, and Beats. Together, Elasticsearch, Logstash, and Kibana became the ELK Stack (now Elastic Stack) and sowed the seeds of a new use case (logging). And now we have a dedicated Observability solution. Then security practitioners took notice and started using the ELK Stack to power their security analytics, and now we have a free and open SIEM. This has continued all these years with community-created and -inspired features, extensions, plugins, and use cases. Free and open is in our DNA Free and open principles are ingrained into who we are and how we progress. We want our products to be used to learn, to develop, and to be run in production at scale. And that’s why many of our core features, products, and solutions are free. For example: Free security features developed natively in the stack are critical to ensure that every cluster is protected. But we go far beyond that, providing role-based access control and true multitenancy for Kibana, all for free. We believe that products developed in the open are more secure. Kibana Lens , for example, which we introduced as a beta in the 7.5 release of the Elastic Stack, made it even easier to visualize data stored in Elasticsearch. It channeled a lot of the community feedback we saw on the Kibana repo over the years. Elastic Maps, which we made generally available in version 7.3 of the Elastic Stack, opened up new ways to visually explore location data in Elasticsearch. This was based on a lot of geo work done in Elasticsearch. Canvas , which became generally available in 6.5, lets you turn your Elasticsearch data into dynamic presentation style dashboards, and bring your unique style to how you tell the story of your data. We’ve even applied this open philosophy to develop turnkey solutions that solve our users’ key challenges. Each of these solutions has components that are built out in the open, and anyone can get started for free. Elastic Enterprise Search makes it possible to implement powerful, modern search experiences complemented by free and open developer tools. Elastic Observability brings together our free and open log monitoring, metrics, APM, and uptime monitoring products into a single powerful solution. Elastic Security combines a free SIEM with an open community, open roadmap, and open data model. These are just a handful of examples. There is so much more to explore. Get started now Everyone has access to a fast and frictionless getting started experience with the Elastic Stack. You can get started with the Elastic Stack in a few different ways. It takes only 3 minutes to spin up a free trial in Elastic Cloud — meaning in less time than it takes to make a cup of coffee you can have your very own Elastic Stack ready to go. Like to test things out locally? No problem! You can always download the latest versions and run them yourself . Want to see the Elastic Stack in action and learn how companies around the world use Elastic products and solutions to tackle challenging business and technology problems? Check out our library of instructional how-to webinars as well as recorded customer case studies . We believe that the best products are built in the open, in collaboration with a passionate group of developers and users who push the bounds of what’s possible. That means we need you! Hop into our forums and engage directly with our engineers or check out what’s happening in our public GitHub repositories . To us, contributing isn’t only about writing code — it’s about bringing new ideas, showing others what’s possible, and creating a community where everyone can learn and grow.",
          "seo": "Being free and open is in our DNA. In the early days, Elasticsearch, Logstash, and Kibana became the ELK Stack (now Elastic Stack) and sowed the seeds of a new use case (logging). And now we have a dedicated Observability solution. Then security practitioners took notice and started using ELK Stack to power their security analytics, and now we have a free and open SIEM. This has continued all these years with community-created and -inspired features, extensions, plugins, and use cases.",
          "url": "https://www.elastic.co/blog/elasticsearch-free-open-limitless",
          "versions": [],
          "category": "blt0c9f31df4f2a7a2b",
          "publish_date": "2020-04-06T17:00:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f551f3c5f2ef1f95a78b66",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Security for Elasticsearch is now free | Elastic Blog",
          "locale": "de-de,fr-fr,ja-jp,ko-kr,zh-cn,pt-br,es-mx,en-us",
          "content": "20 May 2019 Product Security for Elasticsearch is now free By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr We are thrilled to announce that the core security features of the Elastic Stack are now free. This means that users can now encrypt network traffic, create and manage users, define roles that protect index and cluster level access, and fully secure Kibana with Spaces. This is an exciting next step for our community. We opened the code of these features (and many more) last year and by making them free today, everyone can now run a fully secure cluster, hassle free. Security is free, starting in versions 6.8.0 and 7.1.0 For a change this important, we wanted to make sure that it was available to as many people as possible, so today we are releasing versions 6.8.0 and 7.1.0 of the Elastic Stack. These versions do not contain new features; they simply make the following core security features free in the default distribution of the Elastic Stack: TLS for encrypted communications File and native realm for creating and managing users Role-based access control for controlling user access to cluster APIs and indexes; also allows multi-tenancy for Kibana with security for Kibana Spaces Previously, these core security features required a paid Gold subscription. Now they are free as a part of the Basic tier. Note that our advanced security features — from single sign-on and Active Directory/LDAP authentication to field- and document-level security — remain paid features. See the full feature matrix for details . As always, these releases are available immediately on Elasticsearch Service on Elastic Cloud, the official hosted Elasticsearch. But wait, there’s more … heya Kubernetes We are announcing this change in conjunction with the announcement and alpha release of Elastic Cloud on Kubernetes (ECK) , the official Kubernetes Operator for Elasticsearch and Kibana. ECK is designed to automate and simplify how Elasticsearch is deployed and operated in Kubernetes. Security is an integral part of cluster operations, especially in shared and multi-tenant environments like Kubernetes. By moving the core security features into the default distribution of Elastic Stack, we can ensure that all clusters launched and managed by ECK are secured by default at creation time, without any additional burden on the admins. This experience also lines up well with the secure-by-default experience that users have always had on our Elasticsearch Service on Elastic Cloud. Upgrade and get started To get started, download and install the latest version of the Elastic Stack , or upgrade your clusters to either 6.8 or 7.1. To simplify your getting started journey, we have some great material to get you started: A blog about getting started with Elasticsearch security A video walkthrough of getting started with Elasticsearch security Fundamentals of Securing Elasticsearch on-demand training",
          "seo": "We are thrilled to announce that the core security features of the Elastic Stack -- like TLS encryption, RBAC, and both file and native authentication -- are now free.",
          "url": "https://www.elastic.co/blog/security-for-elasticsearch-is-now-free",
          "versions": [
            "6.8"
          ],
          "category": "blt3f90b5g1edce6vd4",
          "publish_date": "2019-05-20T20:02:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f5539fc5f2ef5815ab18bc",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Elastic License Update | Elastic Blog",
          "locale": "en-us",
          "content": "03 June 2021 News Elastic License Update By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr In January 2021, we announced that starting with version 7.11, we would be changing the Apache 2.0 portions of Elasticsearch and Kibana source code to be dual licensed under Elastic License and SSPL, at the users’ discretion. As part of that change, we created Elastic License 2.0 (ELv2) as a permissive, fair-code license, which allows free use, redistribution, modification, and derivative works, with only three simple limitations, outlined in our original announcement . We've been happy to see the level of community support for this change. While we would have preferred to collaborate with Amazon, we're glad that with their fork and commitment to rename the Amazon Elasticsearch Service, it means that they will no longer be the cause of confusion in the Elasticsearch and Kibana community and ecosystem. This is important because we’ve seen so many issues over the years where folks use Amazon's service or other limited forks and expect the products to be the same when they're not. And we're happy to move past the confusion, knowing our users will no longer be misled or have a bad experience because the products are not the same. Since the announcement, Elasticsearch and Kibana have been progressing at a rapid pace, with 3 big releases with new free and open features across the board. And we've also been investing heavily in data shippers, client libraries, and all the other integrations to ensure that they work seamlessly, with good defaults across the full feature set of Elasticsearch. As a result of this, Beats and Logstash 7.13+ now check that they're connecting to an actual Elasticsearch cluster when using the Elasticsearch output. This ensures that the full set of expected capabilities are available and users can rely on them in production. If these capabilities do not exist, Beats, Logstash, and client libraries in future releases will fail early instead of failing or misbehaving in unexpected ways at runtime. As planned and noted during the license change earlier this year, we are now beginning to roll out the more permissive ELv2 to replace Elastic License 1.0 (ELv1) across the rest of our products, including Logstash, Beats, Elastic Agent, APM-Server, and Elastic Cloud on Kubernetes (ECK). To be clear, these changes will only be replacing ELv1 with ELv2, and we are not changing the license of any of the Apache 2.0 licensed code in these products. This simply moves code and distributions that were under ELv1 to ELv2, which is a much simpler and more permissive license. Finally, we are also aligning the license of two tightly coupled Kibana libraries, EUI and Elastic Charts , to the way that Kibana itself is now licensed. These repositories will move from Apache 2.0 to be dual licensed under ELv2 and SSPL. This change, and the transition from ELv1 -> ELv2, will take place over the next several weeks. We want to be clear on our intentions and share these next steps openly. If you have questions, please check out the FAQ , read more about ELv2 , or reach out to us at elastic_license@elastic.co .",
          "seo": "",
          "url": "https://www.elastic.co/blog/elastic-license-update",
          "versions": [],
          "category": "blt0c9f31df4f2a7a2b",
          "publish_date": "2021-06-03T15:00:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f553fbc5f2efa083abcb65",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "The Opening of X-Pack: Phase 1 Complete | Elastic Blog",
          "locale": "en-us",
          "content": "25 April 2018 News The Opening of X-Pack: Phase 1 Complete By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr In the opening keynote of ElasticON 2018, Shay announced that we would be opening the code of X-Pack. You can read more about the motivations in Shay's announcement blog and learn about the details on the dedicated page . Today, we’re excited to announce that we’ve taken the first big step and pushed the code of X-Pack — our security, monitoring, alerting, reporting, graph, and machine learning features — to our public repositories under the Elastic License . So the next time you git pull or browse the source code of Elasticsearch, Kibana, Beats, or Logstash, you will see a new folder - x-pack . Check the contributors guide of each project for any changes to the build process associated with the new code. To be clear, while the X-Pack source code is now available in the public repositories, it isn’t under an Open Source license. The X-Pack source code is governed by the Elastic License , which grants a liberal right to build/test/contribute, but it doesn’t mean that all X-Pack features are free. Some features, like monitoring, search profiler, and the upcoming Index Management UI and Rollup API are free, while others like security and machine learning are available with a paid subscription . One of the aspects of the opening of X-Pack that we’re most excited about is that everyone can now collaborate on public issues together. If there is a chart you wish we showed in monitoring - you can +1 an existing issue, or create your own. This allows us to get more direct feedback during development, which will help us build better features — together. Our next big milestone will be the 6.3 release, where free X-Pack features will be included in the default distribution of the Elastic Stack. For more details and the motivations behind our plans, see Shay’s blog post , and the Opening X-Pack page . If you have questions, you can reach us @elastic on twitter, or find us on the Discuss forums . ​",
          "seo": "",
          "url": "https://www.elastic.co/blog/opening-x-pack-phase-1-complete",
          "versions": [],
          "category": "blt0c9f31df4f2a7a2b",
          "publish_date": "2018-04-25T09:00:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f554a1c5f2ef01f1acf26f",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Elastic now providing distributions for OpenTelemetry SDKs | Elastic Blog",
          "locale": "en-us",
          "content": "Table of Contents Table of contents Elastic now providing distributions for OpenTelemetry SDKs What is OpenTelemetry? A richer instrumentation landscape Elastic and OpenTelemetry What is an OpenTelemetry distribution? The Elastic OpenTelemetry SDK distributions Close Elastic now providing distributions for OpenTelemetry SDKs Adopting OpenTelemetry native standards for instrumenting and observing applications By Steve Gordon 03 April 2024 Share on Twitter Share on Twitter Share on LinkedIn Share on LinkedIn Share on Facebook Share on Facebook Share by Email Share by email Print this page Print If you develop applications, you may have heard about OpenTelemetry . At Elastic®, we are enthusiastic about OpenTelemetry as the future of standardized application instrumentation and observability. In this post, we share our plans to expand our adoption of and commitment to OpenTelemetry with the introduction of Elastic distributions of the OpenTelemetry language SDKs, which will complement our existing Elastic APM agents. What is OpenTelemetry? OpenTelemetry is a vendor-neutral observability framework and toolkit that supports telemetry signals such as traces, metrics, and logs in applications and distributed microservice-based architectures. Driven by a set of standards, OpenTelemetry is designed to provide a consistent approach to instrumenting and observing application behavior. OpenTelemetry is an incubating project developed under the Cloud Native Computing Foundation ( CNCF ) umbrella and is currently the second most active project, topped only by Kubernetes. You can read more on the OpenTelemetry website about the concepts, terminology, and techniques for adopting OpenTelemetry. A richer instrumentation landscape By adopting OpenTelemetry, software code can be instrumented in a vendor-agnostic fashion, with telemetry signals exported in a standardized format to one or more vendor backends, such as Elastic APM . Its design provides flexibility for application owners to switch out vendor backends with no code changes and use OpenTelemetry collectors to send telemetry data to multiple backends. Because OpenTelemetry is not a vendor-specific solution, it is much easier for language ecosystems to adopt it and provide robust instrumentations. Vendors don’t have to implement specific instrumentations themselves anymore. OpenTelemetry is a standard, and it is in the interest of library developers to introduce and maintain instrumentations from which all consumers can benefit. As a result, more instrumentation libraries are available and better kept up to date. If your company has open-source libraries, you can also contribute and create your own instrumentations to make it easier for your customers to adopt OpenTelemetry and benefit from richer traces, metrics, and logging in their applications. Elastic and OpenTelemetry Elastic is deeply involved in OpenTelemetry. In 2023, we donated the Elastic Common Schema , which is being merged with the Semantic Conventions . In 2024, we are in the process of donating our profiling agent based on eBPF . We also have multiple contributors to various areas of OpenTelemetry across the organization. We are therefore committed to helping OpenTelemetry succeed, which means, in some cases, beginning to shift away from Elastic-specific components and recommend using OpenTelemetry components instead. Elastic is committed to supporting and contributing to OpenTelemetry. Our APM solution already accepts native OTLP (OpenTelemetry Protocol) data, and many of our APM agents have already bridged data collection and transmission from applications instrumented using the OpenTelemetry APIs. The next step on our journey is introducing Elastic distributions for the language SDKs and donating features upstream to the OpenTelemetry community by contributing to the OpenTelemetry SDK repositories. What is an OpenTelemetry distribution? An OpenTelemetry distribution is simply a customized version of one or more OpenTelemetry components. Each distribution extends the core functionality offered by the component while adhering to its API and existing features, utilizing built-in extension points. The Elastic OpenTelemetry SDK distributions With the release of Elastic distributions of the OpenTelemetry SDKs, we are extending our backing of OpenTelemetry as the preferred and recommended choice for instrumenting applications. OpenTelemetry maintains and ships many language APIs and SDKs for observing applications using OpenTelemetry. The APIs provide a language-specific interface for instrumenting application code, while the SDK implements that API, enabling signals from observed applications to be collected and exported. Our current work extends the OpenTelemetry language SDKs to introduce additional features and ensure that the exported data provides the most robust compatibility with our current backend while it evolves to become more OpenTelemetry native. Additional features include reimplementing concepts currently available in the Elastic APM Agent but not part of the OpenTelemetry SDK. The distributions allow us to ship with opinionated defaults for all signals that are known to provide the best integration with Elastic’s Observability offering. It’s undoubtedly possible to use the OpenTelemetry APIs to instrument code and then reference the OpenTelemetry SDK to enable the collection of the trace, metric, and log data that applications produce. Elastic APM accepts native OTLP data, so you can configure the OpenTelemetry SDK to export telemetry data directly to an Elastic backend. We refer to this setup as using the “vanilla” (a.k.a. “native”) OpenTelemetry SDK. Work is ongoing to improve support for storing and presenting OpenTelemetry data natively in our backend so that we can drive our observability UIs directly from the data from the various telemetry signals. Our work focuses on ensuring that the Elastic-curated UIs can seamlessly handle the ECS and OpenTelemetry formats. Alongside this effort, we are working on distributions of the language SDKs to support customers looking to adopt OpenTelemetry-native instrumentation in their applications. The current Elastic APM Agents support features such as central configuration and span compression that are not part of the OpenTelemetry specification as of today. We are investing our engineering expertise to bring those features to a broader audience by contributing them to OpenTelemetry. Because standardization takes time, we can more rapidly bring these features to the OpenTelemetry community and our customers by providing distributions. We believe the responsible choice is to concentrate on enabling and encouraging customers to favor vendor-neutral instrumentation in their code and reap the benefits of OpenTelemetry. Distributions best serve our decision to fully adopt and recommend OpenTelemetry as the preferred solution for observing applications. By providing features that are currently unavailable in the “vanilla” OpenTelemetry SDK, we can support customers who want to adopt OpenTelemetry native, vendor-agnostic instrumentation in their applications while still providing the same set of features and backend capabilities they enjoy today with the existing APM Agents. By maintaining Elastic distributions, we can also better support our customers with enhancements and fixes outside of the release cycle of the “vanilla” OpenTelemetry SDKs, which we believe to be a crucial differentiating factor in choosing them. Our vision is that Elastic will work with the OpenTelemetry community to donate features through the standardization processes and contribute the code to implement those in the native OpenTelemetry SDKs. In time, we hope to see many Elastic APM Agent-exclusive features transition into OpenTelemetry to the point where an Elastic distribution may no longer be necessary. In the meantime, we can deliver those capabilities via our OpenTelemetry distributions. Application developers then have several options for instrumenting and collecting telemetry data from their applications: Elastic APM Agent: The most fully featured, however, vendor-specific Elastic APM Agent with OpenTelemetry Bridge: Vendor-neutral instrumentation API, but with known limitations: Only supports bridging of traces (no metrics support) Does not support OpenTelemetry span events OpenTelemetry “vanilla” SDK: Fully supported today; however, it lacks some features of Elastic APM Agent, such as span compression Elastic OpenTelemetry Distribution: Supports vendor-neutral instrumentation and no Elastic-specific configuration in code by default Recommended defaults when using Elastic Observability as a backend Use OpenTelemetry APIs to further customize our defaults; no new APIs to learn While we continue to support all options to instrument your code for the foreseeable future, we think we are setting our customers up for success by introducing a fourth OpenTelemetry-native offering. We expect this will become the preferred default for Elastic customers in due time. We currently have distributions in alpha release status for .NET and Java , with additional language distributions coming very soon. We encourage you to check out those repositories, try out the distributions, and provide feedback to us via issues. Your valued input allows us to refine our designs and steer our direction to ensure that our distributions delight consumers. Learn about the alpha release of our new Elastic distribution of the OpenTelemetry SDK for .NET. The release and timing of any features or functionality described in this post remain at Elastic's sole discretion. Any features or functionality not currently available may not be delivered on time or at all. Share Share on Twitter Share on Twitter Share on LinkedIn Share on LinkedIn Share on Facebook Share on Facebook Share by Email Share by email Print this page Print Sign up for Elastic Cloud free trial Spin up a fully loaded deployment on the cloud provider you choose. As the company behind Elasticsearch , we bring our features and support to your Elastic clusters in the cloud. Start free trial",
          "seo": "",
          "url": "https://www.elastic.co/blog/elastic-opentelemetry-sdk-distributions",
          "versions": [
            "6"
          ],
          "category": "blt3f90b5g1edce6vd4",
          "publish_date": "2024-04-03",
          "authors": [
            {
              "last_name": "Gordon",
              "title": "Steve Gordon",
              "company": "",
              "first_name": "Steve",
              "job_title": "Senior Software Engineer"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f554f1c5f2ef2bb5ad7ae9",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "ElasticON, Elastic’s biggest event of the year, is back and coming to a city near you | Elastic Blog",
          "locale": "en-us",
          "content": "Table of Contents Table of contents ElasticON, Elastic’s biggest event of the year, is back and coming to a city near you Close ElasticON, Elastic’s biggest event of the year, is back and coming to a city near you By Steve Kearns 05 October 2022 Share on Twitter Share on Twitter Share on LinkedIn Share on LinkedIn Share on Facebook Share on Facebook Share by Email Share by email Print this page Print ElasticON is here! Our signature conference series brings the Elastic community — customers, partners, analysts, experts, and more — together to get inspired and learn from one another as we explore new possibilities with Elastic. Elastic leaders will share product roadmaps, expert advice, and information about Elastic's future plans. And customer speakers will join us to discuss their experiences using Elastic for observability, security, and search applications. We’d love to see you there! Where you can attend ElasticON Our ElasticON Global event will take place virtually from March 7–9, 2023. But we’re not waiting until March to engage. We’re taking ElasticON on the road with the ElasticON Comes to You event series in six cities worldwide leading up to our global multi-day virtual event. We kicked off ElasticON virtually from our Bay Area headquarters on October 4, 2022, and will meet in person over the next few months in: Singapore (October 27) New York City (November 2) Amsterdam (November 22) Tokyo (November 30) Washington, D.C. , Public Sector (February 1) The ElasticON experience Our ElasticON Comes to You events will feature in-person keynotes from Elastic leaders who will share how our search-powered solutions empower organizations to solve critical challenges, preview upcoming innovations, and offer a glimpse into the stateless future of the Elastic Search Platform. We’ll also have experts on hand to demo our solutions, offer tips and advice, provide product roadmaps, and share new ways to help you transform your organization. You can choose from two event tracks: technical breakout sessions and real-world use cases. The technical breakout sessions will cover how to: Deliver comprehensive, easy, and modern search experiences combining new vector search capabilities and traditional search technology with Elastic Enterprise Search. Look across your entire deployment to eliminate waste in your production systems with new capabilities in Elastic Observability. Gain broader visibility, drive faster response to threats, and improve your overall speed to security with new updates in Elastic Security. The real-world use cases track will feature speakers from several major companies who will share how Elastic is helping them solve complex, real-world problems by: Building compelling applications for their customers and employees. Observing their infrastructure and applications to reduce downtime and improve efficiency. Protecting their infrastructure and data from cyberattacks. You’ll also have a chance to meet and network with other Elastic community members to swap tips and stories and see how others have solved challenges you may be facing. Check out the agendas for each of our in-person events. Hands-on training opportunities Select ElasticON Comes to You event destinations ( NYC , Amsterdam , and Tokyo ) will host a three-day, instructor-led Elasticsearch Engineer training course. It will show you how to manage and scale your clusters, build custom search applications, and more. If you sign up for training (price varies by location), you’ll receive a complimentary pass to your local ElasticON Comes to You event and one free Elastic Certified Engineer Exam voucher . Join us at ElasticON No matter how or where you attend our upcoming ElasticON events, we’re excited to meet with you and the rest of our fantastic community and share how Elastic can transform your organization. Visit the ElasticON site to explore our event options and locations and to register today! Share Share on Twitter Share on Twitter Share on LinkedIn Share on LinkedIn Share on Facebook Share on Facebook Share by Email Share by email Print this page Print Sign up for Elastic Cloud free trial Spin up a fully loaded deployment on the cloud provider you choose. As the company behind Elasticsearch , we bring our features and support to your Elastic clusters in the cloud. Start free trial",
          "seo": "",
          "url": "https://www.elastic.co/blog/dont-miss-the-elastic-event-of-the-year-elasticon",
          "versions": [
            "12"
          ],
          "category": "blt0c9f31df4f2a7a2b",
          "publish_date": "2022-10-05",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f55343c5f2efcee7aa25e5",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Elastic Stack 7.0.0 released | Elastic Blog",
          "locale": "de-de,fr-fr,ja-jp,ko-kr,zh-cn,pt-br,es-mx,en-us",
          "content": "10 April 2019 Product release Elastic Stack 7.0.0 released By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr 7.0 is here! This release represents more than 10,000 pull requests from 861 committers, so first, a big thank you to our employees and community. If you’d like to hear about the release from the people behind the code, we’ll be hosting a live virtual launch event on April 25, 2019, at 8 am PDT. Join us for 7.0 demos, an AMA with Elastic engineers from around the globe, and more. Elastic Stack 7.0 is immediately available for download , or you can spin up fully managed deployments on the Elasticsearch Service on Elastic Cloud — the only hosted solution to offer new Elastic Stack versions the day they launch. With so much goodness in 7.0, it’s hard to know where to start, so let’s dive right in. Kibana 7.0: new design & navigation … and dark mode! With the Kibana 7.0 design, we decided to place the focus on the content, so you will notice that the UI has a lighter, more minimal feel throughout. The most prominent change is a switch to a new global navigation, which introduces a constant header to switch Kibana spaces, display breadcrumbs, and initiate user actions like change password or logout. To achieve this, and increase consistency, we created the Elastic UI Framework. Over the last year, we converted nearly all of Kibana to use these components. With these components, and a herculean effort from our design and engineering teams, we also made dramatic simplifications to how styles and style sheets are applied. The increased consistency and style sheet improvements enabled us to check off what felt like one of the biggest feature requests in Kibana history — dark mode across all of Kibana. As another benefit of these changes, Kibana dashboards now have a responsive design, which is the first step in dramatically improving usability on mobile devices. A new era for cluster coordination in Elasticsearch Since the beginning, we have focused on making Elasticsearch easy to scale and resilient to catastrophic failures. To support these requirements, we have taken multiple approaches, from making individual nodes more scalable and reliable, to continuous improvement to our cluster coordination layer, known as Zen Discovery. With 7.0, we’re introducing big improvements in both areas yet again. There is a completely new cluster coordination layer for Elasticsearch, which is faster, safer, and easier to use. To achieve this, we started by focusing on the theoretical correctness of our new distributed consensus algorithm using formal models to validate the design . While there are well-known consensus algorithms, like Paxos, Raft, Zab and Viewstamped Replication (VR), the demands of an Elasticsearch cluster require higher throughput for cluster changes, support for easily growing or shrinking a cluster, and a seamless rolling upgrade strategy to allow 6.7 clusters to do a rolling upgrade to 7.0, features that these reference algorithms couldn’t provide. The new cluster coordination layer also includes a number of changes that reduce the likelihood of human error and provides clearer choices when recovering from catastrophic failure. It’s not easy to improve reliability, performance and user experience all at once, especially in such a central component. We’re proud of the new cluster coordination layer, and the process we undertook to get here. To learn more, read the blog . Individual nodes in Elasticsearch are built with resiliency in mind. If you send too many requests to a node or your requests are too large, the node will push back. We achieve this with circuit breakers in Elasticsearch, which determine that the node wouldn’t be able to handle a given request, and immediately respond by asking the client to retry, perhaps on a different node. For nodes with smaller JVM heap sizes, which are becoming more common as users move to a cluster-per-tenant model rather than a massive multitenant cluster, this is even more important. In 7.0, we’re introducing the real memory circuit breaker , which much more accurately detects unserviceable requests, and prevents them from making an individual node unstable. Read the blog to learn more about how this change improves overall node and cluster reliability. Giving relevance and speed a boost across use cases Relevance and speed are the cornerstones of a good search experience. And Elasticsearch 7.0 introduces several foundational features that improve both. Faster top k queries : In many search use cases, quickly seeing the top k (say 20) results on a query matters much more to the user than the exact hit count (i.e., total number of results matching the query). For example, if someone is searching for a product on an e-commerce website they are much more interested in the 10 most relevant results, rather than the other 120,897 results that matched their search query. Elasticsearch 7.0 (and Lucene 8.0) implements a new algorithm (Block-Max WAND) that provides a huge speed boost when retrieving top hits. Intervals queries: Some search use cases, for example, legal and patent search, introduce the need to find records in which words or phrases are within a certain distance from each other. Intervals queries in Elasticsearch 7.0 introduce a brand new way of structuring such queries, and are significantly simpler to use and define compared to the previous method (span queries). Intervals queries are also much more resilient to edge cases compared to span queries. Function score 2.0: Custom scoring is the bread and butter of advanced search use cases, where one would want finer control on relevancy and results ranking. Elasticsearch has provided the ability to do this since its early days. 7.0 introduces the next generation of function score capability that provides a simpler, modular, and more flexible way to generate a ranking score per record. The new modular structure allows users to mix and match a set of arithmetic and distance functions to construct arbitrary function score calculations, giving them more control over how results are scored and ranked. Smooth zoom in Elastic Maps with geotile grid Over the years, our support for geo data has continuously improved — from the early days when geo support was first added to Elasticsearch , to introducing the Bkd-Tree data structure to Lucene and using it to improve geoshape query performance by over 25x, to the Elastic Maps service that powers the global basemap in Kibana. With 7.0, we continue this investment, introducing a new aggregation in Elasticsearch to handle (geo) map tiles in a way that allows a user to zoom in and out on the map without any change to the shape of the result data. The new geotile grid aggregation groups geo points into buckets that represent cells in a grid, with each cell correlating with a tile in a map. Prior to this change, the fringes of the shape could slightly change with the change in zoom level, because the rectangular tiles would change orientation at different zoom levels. Elastic Maps in 7.0 is already using this new aggregation to ensure that your view stays stable as you zoom in and out. This level of accuracy is important, whether you're protecting your network from attackers, investigating slow application response times in specific locations, or tracking your brother hiking the Pacific Crest Trail . Strengthening time series use cases with nanosecond precision support Whether it’s infrastructure metrics, system audit logs, network traffic, or a rover on Mars, time series data is central to how many people use the Elastic Stack. The ability to precisely order and correlate events across multiple systems and services is key. Up until now, Elasticsearch only stored timestamps with millisecond precision. 7.0 adds a few zeroes and brings nanosecond precision, which allows users with high-frequency data collection needs the precision required to accurately store and sequence this data. The change was made possible by migrating from the historical JODA library to the official Java time API in JDK 8.",
          "seo": "Version 7.0 of the Elastic Stack lands with a fresh design and pervasive dark mode in Kibana, a rebuilt cluster coordination layer, and boosts to query speeds & relevance across use cases.",
          "url": "https://www.elastic.co/blog/elastic-stack-7-0-0-released",
          "versions": [
            "7.0"
          ],
          "category": "bltfaae4466058cc7d6",
          "publish_date": "2019-04-10T16:06:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f551fbc5f2ef411ba7992a",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Elastic Stack 6.6.0 Released | Elastic Blog",
          "locale": "de-de,fr-fr,ja-jp,ko-kr,zh-cn,es-mx,en-us",
          "content": "29 January 2019 Product release Elastic Stack 6.6.0 Released By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr 6.6.0 has arrived! This release has new features across the stack that simplify how you manage and scale your cluster, faster geoshape indexing and querying with more efficient storage, and key improvements to Elasticsearch SQL, machine learning, Auditbeat, and more! Deploy a cluster on our Elasticsearch Service or download the stack to take these latest features for a spin. Manage Data Lifecycle at Scale with Index Lifecycle Management Users with time series use cases like logging, metrics, and APM, typically store data in time-based indexes. As this data ages, there are a number of ways to ensure it’s being stored in the most cost-effective way. For example, as the index ages, the user might want to shrink the number of shards or reduce the number of replicas used to store the index, or move it to nodes deployed on cheaper hardware. Or they might want to delete indices that are older than a certain age. Existing methods for defining policies to manage the lifecycle of the index live outside the cluster (for example, Curator or custom automation scripts), are limited, and introduce management overhead to configure and monitor. The new index lifecycle management feature provides a more integrated and streamlined way to manage this data, making it easier to live with best practices. The index lifecycle management feature breaks the lifecycle of an index into four phases: hot, warm, cold, and delete phase. You can define an index lifecycle policy which allows you to: Have one primary shard on each hot node to maximize indexing throughput. Replace the hot index with a new empty index as soon as the existing index is “full” or after a time period. Move the old index to warm nodes, where it can be shrunk to a single shard and force-merged down to a single segment for optimized storage and querying. Later, move the index to cold nodes for cheaper storage. In future release, you’ll be able to “freeze” the index, putting it in a state which trades storage density for search latency. And finally, delete the index once it is no longer useful to you. All of this is handled for you automatically by index lifecycle management. Frozen Indices Enable Higher Storage to Memory Ratios Elasticsearch is highly optimized to perform searches as quickly and efficiently as possible. So historically, each open (read: searchable) index used a small amount of memory to ensure that any query hitting that index would execute fast. The bigger the size and number of indexes on a given node, the more memory required to keep indexes in this open state. This effectively means that there are practical limits to the amount of storage a given node would be able to address with a single JVM. For most users and use-cases, this is not an issue. However in some cases, like those requiring long term archive of multiple years of data for regulatory reasons, there is a desire to keep the data online and searchable, but less of a need for peak performance on requests over the older data. Frozen indices allow for a much higher ratio of disk storage to heap, at the expense of search latency. When an index is frozen, it takes up no heap, allowing a single node to easily manage thousands of indices with very low overhead. When a search targets frozen indices, the query will fully open, search, and then close each index sequentially. Frozen indices are replicated, unlike closed indexes. Frozen indices provide a new set of choices for how to optimize your cluster cost and performance around your needs. Faster and Smaller Bkd-Backed Geoshapes The Bkd tree data structure keeps delivering. Back in 5.0, we introduced Bkd-backed geopoints, which resulted in significant storage, memory and performance improvements for querying geopoints. With 6.6.0, we bring the same Bkd-based benefits to geoshapes! We’ve achieved the triple-crown of search - Indexing is faster, it will take up less space on disk, and will use less memory. Elasticsearch SQL Adds Support for Date Histograms Elasticsearch SQL continues to march toward GA with a slew of improvements addressing time queries, including native support for date histograms using SQL syntax. These improvements are great for all users of Elasticsearch SQL, but we expect that date histogram support will be most impactful for Canvas users, making it easier to build time series charts in Kibana. Machine Learning Introduces Annotations When investigating a potential system or security issue, it’s natural to want to record your findings and progress - recording the root cause of a system issue and steps taken to resolve, etc. Now, directly inside the machine learning UI, you can create annotations that all users can see. This simplifies collaboration and allows you to keep a record of actions taken, without leaving Kibana. Elastic APM Adds New Agent Metrics APM is introducing agent metrics in the 6.6 release. The latest version of our agents will now automatically report system and process-level CPU and memory metrics, along with traces and errors. In other news, Distributed Tracing is now generally available, and all agents are OpenTracing compliant. Lastly, the APM UI is making it effortless to jump from APM to relevant Logging or Infrastructure views, and the Java agent has introduced two new great features. Read the APM blog post for all the details. In addition to these changes, we are also excited to announce that Elastic APM will now be available to deployments on Elasticsearch Service ( read blog ) and Elastic Cloud Enterprise ( read blog ). But wait, there’s more … In addition to all these, we also added several new features and improvements in Beats, Logstash, and Kibana. Auditbeat added a new system module to collect various security related information from the system. This includes data about the operating system, processes, sockets and users existing on a particular host. You can read more about the Auditbeat system module in the dedicated blog post . Machine learning now comes prepackaged with machine learning jobs for Auditbeat data, giving users a jump start on detecting common anomalies in their audit data. Filebeat adds a new NetFlow input, which can be used to receive these Netflow and IPFIX records over UDP. It supports NetFlow v1, v5, v6, v7, v8, v9 and IPFIX. When using Beats Central Management, you can now configure Metricbeat and Filebeat to reject modifications to some parts of their configuration. This allows effective enforcement at the running beat level, of what can be modified by the remote configuration. To enhance secure operation, we now block modifications to the console output and the file output sections by default. On the Logstash side, the more performant Java execution engine, that was introduced as a beta in 6.1, is now generally available. On the Kibana side, we introduced a highly requested feature that allows a single Kibana instance to connect to multiple Elasticsearch nodes, which circumvents the previous challenges with having a single point of failure on the Kibana <> Elasticsearch connection. On the visualization front, Kibana dashboards can now be exported as PNGs. You can read the Kibana 6.6 release highlights for details on these and other changes. Try it now Deploy a cluster on our Elasticsearch Service or download the stack to take these latest features for a spin.",
          "seo": "Elastic Stack 6.6.0 is here. Check out frozen indices, index lifecycle management, new Bkd-tree back geoshape, and much more.",
          "url": "https://www.elastic.co/blog/elastic-stack-6-6-0-released",
          "versions": [
            "6.6"
          ],
          "category": "bltfaae4466058cc7d6",
          "publish_date": "2019-01-29T18:00:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      },
      {
        "_index": "blogs",
        "_id": "66f552e2c5f2ef4bdca94930",
        "_score": 5.303483,
        "_ignored": [
          "content.keyword"
        ],
        "_source": {
          "title": "Beats 7.5.0 released | Elastic Blog",
          "locale": "en-us",
          "content": "02 December 2019 Product release Beats 7.5.0 released By Steve Kearns Share Share on Twitter Share on Facebook Share on LinkedInr We’re pleased to announce the general availability of the Beats 7.5.0 release. This is the latest stable release and is now available for download ! Please refer to the release notes for the complete list of bug fixes and features. Beats are central to any organization’s observability story, and in recent versions, we’ve been focusing on building turnkey data integrations for the most important infrastructure and service metrics, including Kubernetes, Prometheus, and Amazon Web Services (AWS). In 7.5, we build on that momentum by introducing turnkey monitoring of Microsoft Azure metrics and logs as part of our partnership with Microsoft, and even more improvements for monitoring Kubernetes. Azure cloud monitoring Earlier in the year, we introduced support for Microsoft Azure as part of the Elasticsearch Service on Elastic Cloud. In 7.5, our Azure monitoring story gets even stronger with the addition of turnkey metrics and logs ingestion directly from Azure services. With the addition of Metricbeat and Filebeat modules for Azure monitoring, Azure users can now directly monitor logs and metrics from Azure Event Hub and Azure Monitor and use prebuilt Kibana dashboards to speed up the analysis. Monitoring status of Kubernetes services Kubernetes monitoring is particularly challenging due to the dynamic nature of infrastructure and services running on it. To ensure all these services are available and responding requires really flexible configuration options. In 7.5, we’re enhancing Heartbeat, as part of our Uptime solution, to include hint-based auto-discovery, which is a particularly great fit for monitoring the health of Kubernetes services. There’s more! Interested in our broader observability story? Learn about all the updates across Logs, Metrics, APM, and Uptime in our Elastic Observability post. Ready to get started? Please download Beats 7.5.0 , try it out, and let us know what you think on Twitter ( @elastic ) or in our forums . You can report any bugs or feature requests on the Beats Github issues page .",
          "seo": "In the Beats 7.5 release, we introduce turnkey monitoring of Microsoft Azure metrics and logs as part of our partnership with Microsoft, and even more improvements for monitoring Kubernetes.",
          "url": "https://www.elastic.co/blog/beats-7-5-0-released",
          "versions": [
            "7.5"
          ],
          "category": "bltfaae4466058cc7d6",
          "publish_date": "2019-12-02T17:08:00.000Z",
          "authors": [
            {
              "last_name": "Kearns",
              "title": "Steve Kearns",
              "company": "Elastic",
              "first_name": "Steve",
              "job_title": "Vice President, Product Management"
            }
          ]
        }
      }
    ]
  }
}

```

This step demonstrates that text fields are case-insensitive by default. Both the indexed content and your query term are lowercased by the analyzer before comparison, so Steve, steve, and STEVE all match the same documents. This behavior makes full-text search more forgiving for users.

```bash
GET _analyze
{
  "text": "United Kingdom",
  "analyzer": "standard"
}
```

```
{
  "tokens": [
    {
      "token": "united",
      "start_offset": 0,
      "end_offset": 6,
      "type": "<ALPHANUM>",
      "position": 0
    },
    {
      "token": "kingdom",
      "start_offset": 7,
      "end_offset": 14,
      "type": "<ALPHANUM>",
      "position": 1
    }
  ]
}

```

```bash
GET _analyze
{
  "text": "Nodes and Shards",
  "analyzer": "standard"
}
```

```
{
  "tokens": [
    {
      "token": "nodes",
      "start_offset": 0,
      "end_offset": 5,
      "type": "<ALPHANUM>",
      "position": 0
    },
    {
      "token": "and",
      "start_offset": 6,
      "end_offset": 9,
      "type": "<ALPHANUM>",
      "position": 1
    },
    {
      "token": "shards",
      "start_offset": 10,
      "end_offset": 16,
      "type": "<ALPHANUM>",
      "position": 2
    }
  ]
}


```bash
GET _analyze
{
  "text": "Nodes and Shards",
  "analyzer": "english"
}

```
{
  "tokens": [
    {
      "token": "node",
      "start_offset": 0,
      "end_offset": 5,
      "type": "<ALPHANUM>",
      "position": 0
    },
    {
      "token": "shard",
      "start_offset": 10,
      "end_offset": 16,
      "type": "<ALPHANUM>",
      "position": 2
    }
  ]
}


```bash
POST sample_blog/_doc
{
  "@timestamp": "2021-03-10T16:00:00.000Z",
  "abstract": "The Joy of Painting",
  "author": "Bob Ross",
  "body": "Painting should do one thing. It should put happiness in your heart. We'll take a little bit of Van Dyke Brown. Isn't that fantastic? You can just push a little tree out of your brush like that. Mix your color marbly don't mix it dead.",
  "body_word_count": 55,
  "category": "Painting",
  "title": "Making Happy Little Trees",
  "url": "/blog/happy-little-trees",
  "published": true
}
```

```
{
  "_index": "sample_blog",
  "_id": "_4bsKZ0BoWdhtcXA787r",
  "_version": 1,
  "result": "created",
  "_shards": {
    "total": 2,
    "successful": 1,
    "failed": 0
  },
  "_seq_no": 0,
  "_primary_term": 1
}


```bash
GET sample_blog/_mapping
```

```
{
  "sample_blog": {
    "mappings": {
      "properties": {
        "@timestamp": {
          "type": "date"
        },
        "abstract": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "author": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "body": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "body_word_count": {
          "type": "long"
        },
        "category": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "published": {
          "type": "boolean"
        },
        "title": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "url": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        }
      }
    }
  }
}



```bash
PUT test_blogs
{
  "mappings": {
    "properties": {
      "@timestamp": {
        "type": "date"
      },
      "abstract": {
        "type": "text"
      },
      "author": {
        "type": "keyword"
      },
      "body": {
        "type": "text"
      },
      "body_word_count": {
        "type": "integer"
      },
      "category": {
        "type": "keyword"
      },
      "title": {
        "type": "text"
      },
      "url": {
        "type": "keyword"
      },
      "published": {
        "type": "boolean"
      }
    }
  }
}
```

```{
  "sample_blog": {
    "mappings": {
      "properties": {
        "@timestamp": {
          "type": "date"
        },
        "abstract": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "author": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "body": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "body_word_count": {
          "type": "long"
        },
        "category": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "published": {
          "type": "boolean"
        },
        "title": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "url": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        }
      }
    }
  }
}

````

```bash
# Index a doc after an index has been created
POST test_blogs/_doc
{
  "@timestamp": "2021-03-10T16:00:00.000Z",
  "abstract": "The Joy of Painting",
  "author": "Bob Ross",
  "body": "Painting should do one thing. It should put happiness in your heart. We'll take a little bit of Van Dyke Brown. Isn't that fantastic? You can just push a little tree out of your brush like that. Mix your color marbly don't mix it dead.",
  "body_word_count": 55,
  "category": "Painting",
  "title": "Making Happy Little Trees",
  "url": "/blog/happy-little-trees",
  "published": true
}
```

```
{
  "_index": "test_blogs",
  "_id": "v4b7KZ0BoWdhtcXA9c_q",
  "_version": 1,
  "result": "created",
  "_shards": {
    "total": 2,
    "successful": 1,
    "failed": 0
  },
  "_seq_no": 0,
  "_primary_term": 1
}

```

```bash
PUT blogs_fixed/_mapping
{
  "properties": {
    "authors": {
      "properties": {
        "company": {
          "type": "keyword"
        },
        "first_name": {
          "type": "keyword"
        },
        "job_title": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "last_name": {
          "type": "keyword"
        },
        "title": {
          "type": "text"
        }
      }
    },
    "category": {
      "type": "keyword"
    },
    "content": {
      "type": "text"
    },
    "locale": {
      "type": "keyword"
    },
    "publish_date": {
      "type": "date",
      "format": "iso8601"
    },
    "seo": {
      "type": "text"
    },
    "title": {
      "type": "text",
      "fields": {
        "keyword": {
          "type": "keyword",
          "ignore_above": 256
        }
      }
    },
    "url": {
      "type": "keyword"
    },
    "versions": {
      "type": "keyword"
    }
  }
}

```

{
  "acknowledged": true
}


GET blogs_fixed/_search
{
  "query": {
    "match": {
      "authors.first_name": "Kim"
    }
  }
}

```bash
PUT blogs_fixed2
{
  "mappings": {
    "properties": {
      "search_authors": {
        "type": "text"
      },
      "authors": {
        "properties": {
          "company": {
            "type": "keyword",
            "copy_to": "search_authors"
          },
          "first_name": {
            "type": "keyword"
          },
          "job_title": {
            "type": "text",
            "copy_to": "search_authors",
            "fields": {
              "keyword": {
                "type": "keyword",
                "ignore_above": 256
              }
            }
          },
          "last_name": {
            "type": "keyword"
          },
          "title": {
            "type": "text",
            "copy_to": "search_authors"
          }
        }
      },
      "category": {
        "type": "keyword"
      },
      "content": {
        "type": "text",
        "analyzer": "english"
      },
      "locale": {
        "type": "keyword"
      },
      "publish_date": {
        "type": "date",
        "format": "iso8601"
      },
      "seo": {
        "type": "text",
        "analyzer": "english"
      },
      "title": {
        "type": "text",
        "analyzer": "english",
        "fields": {
          "keyword": {
            "type": "keyword",
            "ignore_above": 256
          }
        }
      },
      "url": {
        "type": "keyword",
        "doc_values": false
      },
      "versions": {
        "type": "keyword"
      }
    }
  }
}
